# Notebook 8 - Run 4 Results Analysis
Evaluates Run 4 results against Run 3 baseline (L3 62.8%).
New in Run 4: Pass 1 DynamoDB lookup on merchant_name, Pass 2 LLM NER + DynamoDB.

**Input:** `outputs/run4_results_20260312_0631.txt` (CSV format)

**Field naming convention (Run 5 onwards):**
- `naive_key`           - LLM, no hints, baseline
- `p1_bank_db_key`      - DB lookup on bank-supplied merchant_name
- `p2_ner_db_key`       - DB lookup on NER-extracted merchant name
- `p2_ner_llm_key`      - LLM category from NER step, no DB (Run 5+, not in Run 4)
- `llm_final_pred_key`  - Full pipeline LLM output, primary metric
- `gt_key`              - Ground truth

**Sections:**
1. Setup and data load
2. Headline metrics
3. L1 / L2a / L3 vs Run 3 baseline
4. Naive vs pipeline lift
5. Pass 1 coverage and accuracy
6. Pass 2 coverage and accuracy
7. Combined P1/P2 coverage
8. Execution path breakdown
9. Signal conflict analysis
10. Confidence vs accuracy
11. Per-category accuracy
12. Per-provider accuracy
13. Four-field accuracy comparison
14. Ad-hoc query helpers

---
## 1. Setup and data load

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 80)

OUTPUT_DIR = Path('outputs')
ASSET_DIR  = Path('assets')

RUN4_FILE     = OUTPUT_DIR / 'run4_results_20260312_0631.csv'
CAT_KEYS_FILE = ASSET_DIR  / 'cat_keys.csv'
TAXONOMY_FILE = ASSET_DIR  / 'taxonomy_master_updated_override.csv'

# Run 3 baseline - L3 only is comparable (L1/L2a used indirect taxonomy join method)
RUN3_L3   = 62.8
RUN3_LIFT = 6.2

print('Imports OK')

In [ ]:
df = pd.read_csv(RUN4_FILE)
print(f'Loaded {len(df):,} rows, {len(df.columns)} columns')
print(df.columns.tolist())

In [ ]:
# Rename Run 4 field names to agreed Run 5 naming convention
# Non-destructive - renames in-memory only, original file unchanged
RENAME_MAP = {
    'base_category_key_naive': 'naive_key',
    'p1_db_category':          'p1_bank_db_key',
    'p2_db_category':          'p2_ner_db_key',
    'base_category_key':       'llm_final_pred_key',
    'gt_base_category_key':    'gt_key',
}
df = df.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df.columns})
print('Columns after rename:')
print(df.columns.tolist())

In [ ]:
# Load valid keys
cat_keys_df = pd.read_csv(CAT_KEYS_FILE)
key_col     = 'base_category_key' if 'base_category_key' in cat_keys_df.columns else 'key'
VALID_KEYS  = set(cat_keys_df[key_col].dropna().str.strip())
print(f'Valid keys: {len(VALID_KEYS)}')

# Load taxonomy
tax = pd.read_csv(TAXONOMY_FILE)
tax.columns = tax.columns.str.strip()
tax = tax.drop_duplicates(subset='base_category_key', keep='first')
print(f'Taxonomy: {len(tax)} keys')

In [ ]:
# Normalise types
df['confidence']  = pd.to_numeric(df['confidence'], errors='coerce')
df['parse_error'] = df['parse_error'].fillna(False).astype(bool)
df['invalid_key'] = df['invalid_key'].fillna(False).astype(bool)

# Re-enforce invalid_key from VALID_KEYS
pred = df['llm_final_pred_key'].fillna('')
df['invalid_key'] = (~pred.isin(VALID_KEYS)) & (~df['parse_error'])

# Parse flag JSON
def parse_flag(v):
    if pd.isna(v) or v in ('', 'null', 'None'): return None
    try:    return json.loads(v)
    except: return None

df['flag_parsed'] = df['flag'].apply(parse_flag)
df['flag_type']   = df['flag_parsed'].apply(lambda x: x.get('type')   if x else None)
df['flag_detail'] = df['flag_parsed'].apply(lambda x: x.get('detail') if x else None)

# Define hit masks - 'na' is the miss sentinel
p1_hit = df['p1_bank_db_key'].notna() & (df['p1_bank_db_key'] != 'na')
p2_hit = df['p2_ner_db_key'].notna()  & (df['p2_ner_db_key']  != 'na')
p1_miss_p2_hit = ~p1_hit & p2_hit

# Evaluable subset
df_eval = df[~df['parse_error'] & ~df['invalid_key']].copy()
print(f'Evaluable rows: {len(df_eval):,} / {len(df):,}')

---
## 2. Headline metrics

In [ ]:
n           = len(df)
n_parse_err = df['parse_error'].sum()
n_invalid   = df['invalid_key'].sum()

# L1
df_l1  = df_eval[df_eval['gt_spend_type'].isin(['spend','non_spend']) & df_eval['spend_type'].notna()]
l1_acc = (df_l1['spend_type'] == df_l1['gt_spend_type']).mean() * 100

# L2a
df_l2a  = df_eval[df_eval['gt_category_type'].notna() & (df_eval['gt_category_type'] != '') & df_eval['category_type'].notna()]
l2a_acc = (df_l2a['category_type'] == df_l2a['gt_category_type']).mean() * 100 if len(df_l2a) else 0

# L3
df_l3  = df_eval.dropna(subset=['gt_key', 'llm_final_pred_key'])
l3_acc = (df_l3['llm_final_pred_key'] == df_l3['gt_key']).mean() * 100

# Naive vs pipeline lift
df_lift     = df_eval.dropna(subset=['gt_key', 'llm_final_pred_key', 'naive_key'])
pipeline_l3 = (df_lift['llm_final_pred_key'] == df_lift['gt_key']).mean() * 100
naive_l3    = (df_lift['naive_key']           == df_lift['gt_key']).mean() * 100
lift        = pipeline_l3 - naive_l3

print('=' * 60)
print('RUN 4 RESULTS SUMMARY')
print('=' * 60)
print(f'Total rows:        {n:,}')
print(f'Parse errors:      {n_parse_err} ({n_parse_err/n*100:.1f}%)')
print(f'Invalid keys:      {n_invalid} ({n_invalid/n*100:.1f}%)')
print(f'Evaluable rows:    {len(df_eval):,}')
print()
print(f'L1 spend_type:     {l1_acc:.1f}%  n={len(df_l1)}')
print(f'  NOTE: Run 3 L1 (85.1%) used indirect taxonomy join - not comparable')
print(f'L2a category_type: {l2a_acc:.1f}%  n={len(df_l2a)}')
print(f'  NOTE: Run 3 L2a (87.9%) used indirect taxonomy join - not comparable')
print(f'L3 exact key:      {l3_acc:.1f}%  n={len(df_l3)}  (Run 3: {RUN3_L3}%  delta: {l3_acc-RUN3_L3:+.1f}pp)')
print()
print(f'naive_key L3:      {naive_l3:.1f}%')
print(f'Pipeline lift:     {lift:+.1f}pp  (Run 3: +{RUN3_LIFT}pp)')

---
## 3. L1 / L2a / L3 vs Run 3 baseline

L1/L2a columns flagged as non-comparable - Run 3 used indirect taxonomy join; Run 4 uses direct GT labels.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
metrics   = ['L1 spend_type\n(GT method differs)', 'L2a category_type\n(GT method differs)', 'L3 exact key\n(comparable)']
run3_vals = [85.1, 87.9, RUN3_L3]
run4_vals = [l1_acc, l2a_acc, l3_acc]
x, w = np.arange(3), 0.35
for i in range(3):
    c3 = '#bdc3c7' if i < 2 else '#95a5a6'
    c4 = '#abebc6' if i < 2 else '#2ecc71'
    ax.bar(x[i]-w/2, run3_vals[i], w, color=c3, label='Run 3' if i==0 else '_')
    ax.bar(x[i]+w/2, run4_vals[i], w, color=c4, label='Run 4' if i==0 else '_')
    ax.text(x[i]-w/2, run3_vals[i]+0.5, f'{run3_vals[i]:.1f}%', ha='center', fontsize=8)
    ax.text(x[i]+w/2, run4_vals[i]+0.5, f'{run4_vals[i]:.1f}%', ha='center', fontsize=8)
ax.axvspan(-0.5, 1.5, alpha=0.04, color='red')
ax.text(0.5, 97, 'not directly comparable', ha='center', fontsize=8, color='red', style='italic')
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=9)
ax.set_ylim(0, 105); ax.set_ylabel('Accuracy (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
ax.set_title('Run 4 vs Run 3 baseline', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb8_accuracy_vs_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Naive vs pipeline lift

In [ ]:
n_correct = (df_l3['llm_final_pred_key'] == df_l3['gt_key']).sum()
n_wrong   = len(df_l3) - n_correct

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['naive_key', 'llm_final_pred_key'], [naive_l3, pipeline_l3], color=['#e67e22','#2ecc71'], width=0.5)
for i, v in enumerate([naive_l3, pipeline_l3]):
    axes[0].text(i, v+0.5, f'{v:.1f}%', ha='center', fontsize=10)
axes[0].set_ylabel('L3 Accuracy (%)')
axes[0].set_ylim(0, 100)
axes[0].set_title(f'Pipeline lift: {lift:+.1f}pp  (Run 3: +{RUN3_LIFT}pp)', fontweight='bold')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))

cats  = ['Correct', 'Wrong (valid)', 'Invalid key', 'Parse error']
vals  = [n_correct, n_wrong, n_invalid, n_parse_err]
clrs  = ['#2ecc71', '#e67e22', '#e74c3c', '#9b59b6']
axes[1].barh(cats, vals, color=clrs, height=0.5)
for i, v in enumerate(vals):
    axes[1].text(v+2, i, f'{v:,}  ({v/n*100:.1f}%)', va='center', fontsize=9)
axes[1].set_xlim(0, n*1.3)
axes[1].set_title('Result breakdown', fontweight='bold')
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb8_lift_and_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Pass 1 (p1_bank_db_key) - coverage and accuracy

In [ ]:
df_p1_attempted = df[df['merchant_name'].notna() & (df['merchant_name'].str.strip() != '')]
df_p1_hit       = df[p1_hit]
p1_attempt_rate = len(df_p1_attempted) / n * 100
p1_hit_rate     = len(df_p1_hit) / len(df_p1_attempted) * 100 if len(df_p1_attempted) else 0

df_p1_eval   = df_p1_hit.dropna(subset=['gt_key'])
p1_db_acc    = (df_p1_eval['p1_bank_db_key'] == df_p1_eval['gt_key']).mean() * 100 if len(df_p1_eval) else 0

df_p1_agree  = df_p1_hit.dropna(subset=['llm_final_pred_key'])
p1_llm_agree = (df_p1_agree['llm_final_pred_key'] == df_p1_agree['p1_bank_db_key']).mean() * 100 if len(df_p1_agree) else 0
p1_llm_acc   = (df_p1_agree['llm_final_pred_key'] == df_p1_agree['gt_key']).mean() * 100 if len(df_p1_agree) else 0

print('PASS 1 - p1_bank_db_key')
print('-' * 50)
print(f'merchant_name populated (P1 attempted): {len(df_p1_attempted):,}  ({p1_attempt_rate:.1f}%)')
print(f'P1 DB hit:                              {len(df_p1_hit):,}  ({p1_hit_rate:.1f}% of attempted)')
print(f'p1_bank_db_key accuracy vs gt_key:      {p1_db_acc:.1f}%  (n={len(df_p1_eval)})')
print(f'llm_final_pred_key on P1 hit rows:      {p1_llm_acc:.1f}%  (delta vs DB: {p1_llm_acc-p1_db_acc:+.1f}pp)')
print(f'LLM agreed with P1 hint:                {p1_llm_agree:.1f}%')

In [ ]:
# Where LLM overrode P1 and was wrong
df_p1_override = df_p1_agree[
    (df_p1_agree['llm_final_pred_key'] != df_p1_agree['p1_bank_db_key'])
].dropna(subset=['gt_key'])

if len(df_p1_override):
    llm_right = (df_p1_override['llm_final_pred_key'] == df_p1_override['gt_key']).sum()
    db_right  = (df_p1_override['p1_bank_db_key']     == df_p1_override['gt_key']).sum()
    print(f'LLM overrode p1_bank_db_key on {len(df_p1_override)} rows:')
    print(f'  LLM correct, DB wrong: {llm_right}')
    print(f'  DB correct, LLM wrong: {db_right}')
    print(f'  Both wrong:            {len(df_p1_override)-llm_right-db_right}')
    print()
    wrong = df_p1_override[df_p1_override['p1_bank_db_key'] == df_p1_override['gt_key']]
    print('Sample rows - DB correct, LLM overrode wrongly:')
    print(wrong[['original_description','merchant_name','p1_bank_db_key','llm_final_pred_key','gt_key','reason']].head(10).to_string(index=False))

---
## 6. Pass 2 (p2_ner_db_key) - coverage and accuracy

RUN_NER_ALWAYS=True in Run 4 - NER ran on every row. P2 NER trigger rate is therefore not a useful production coverage metric; focus on P2 incremental value (P1 miss + P2 hit).

In [ ]:
df_p2_ner_ran = df[df['p2_llm_merchant'].notna()]
df_p2_hit     = df[p2_hit]
p2_ner_rate   = len(df_p2_ner_ran) / n * 100
p2_db_hit_pct = len(df_p2_hit) / n * 100

df_p2_eval   = df_p2_hit.dropna(subset=['gt_key'])
p2_db_acc    = (df_p2_eval['p2_ner_db_key'] == df_p2_eval['gt_key']).mean() * 100 if len(df_p2_eval) else 0

df_p2_agree  = df_p2_hit.dropna(subset=['llm_final_pred_key'])
p2_llm_agree = (df_p2_agree['llm_final_pred_key'] == df_p2_agree['p2_ner_db_key']).mean() * 100 if len(df_p2_agree) else 0
p2_llm_acc   = (df_p2_agree['llm_final_pred_key'] == df_p2_agree['gt_key']).mean() * 100 if len(df_p2_agree) else 0

p2_incr_n    = p1_miss_p2_hit.sum()
df_p2_incr   = df[p1_miss_p2_hit].dropna(subset=['gt_key'])
p2_incr_acc  = (df_p2_incr['p2_ner_db_key'] == df_p2_incr['gt_key']).mean() * 100 if len(df_p2_incr) else 0

print('PASS 2 - p2_ner_db_key (RUN_NER_ALWAYS=True)')
print('-' * 55)
print(f'NER ran (p2_llm_merchant extracted):    {len(df_p2_ner_ran):,}  ({p2_ner_rate:.1f}% - all rows, comparison mode)')
print(f'P2 DB hit:                              {len(df_p2_hit):,}  ({p2_db_hit_pct:.1f}% of all rows)')
print(f'p2_ner_db_key accuracy vs gt_key:       {p2_db_acc:.1f}%  (n={len(df_p2_eval)})')
print(f'llm_final_pred_key on P2 hit rows:      {p2_llm_acc:.1f}%  (delta vs DB: {p2_llm_acc-p2_db_acc:+.1f}pp)')
print(f'LLM agreed with P2 hint:                {p2_llm_agree:.1f}%')
print()
print(f'P2 incremental (P1 miss + P2 hit):      {p2_incr_n} rows')
print(f'  p2_ner_db_key accuracy on incr rows:  {p2_incr_acc:.1f}%')

---
## 7. Combined P1/P2 coverage

In [ ]:
either_hit = (p1_hit | p2_hit).sum()
p1_only    = (p1_hit & ~p2_hit).sum()
p2_only    = (~p1_hit & p2_hit).sum()
both_hit   = (p1_hit & p2_hit).sum()
neither    = (~p1_hit & ~p2_hit).sum()

print(f'Combined DB coverage (P1 or P2 hit):    {either_hit:,}  ({either_hit/n*100:.1f}%)')
print(f'  P1 hit only:                          {p1_only:,}')
print(f'  Both P1 and P2 hit:                   {both_hit:,}')
print(f'  P2 hit only (incremental):             {p2_only:,}  <- pure P2 contribution')
print(f'  Neither hit (LLM reasoning only):     {neither:,}  ({neither/n*100:.1f}%)')
print()
print(f'Estimated production P2 trigger volume: ~{p1_only + neither:,} rows  (if RUN_NER_ALWAYS=False)')

---
## 8. Execution path breakdown

In [ ]:
print('Top 20 execution paths:')
print(df['execution_path'].value_counts().head(20).to_string())
print()
for step in ['biller','p1_db','mcc','dt','merchant','regex','p2_db','llm']:
    n_rows = df['execution_path'].fillna('').str.contains(step).sum()
    print(f'  {step:<12}: {n_rows:,}  ({n_rows/n*100:.1f}%)')

In [ ]:
# L3 accuracy by DB signal source
df_ep = df_eval.dropna(subset=['gt_key','llm_final_pred_key']).copy()
df_ep['l3_correct'] = df_ep['llm_final_pred_key'] == df_ep['gt_key']
for label, mask in [
    ('P1 hit',             p1_hit),
    ('P2 only (P1 miss)',  p1_miss_p2_hit),
    ('No DB hit',          ~p1_hit & ~p2_hit),
]:
    grp = df_ep[mask.reindex(df_ep.index, fill_value=False)]
    if len(grp):
        print(f'  {label:<25}: {grp["l3_correct"].mean()*100:.1f}%  (n={len(grp)})')

---
## 9. Signal conflict analysis

In [ ]:
print('Flag type distribution:')
print(df['flag_type'].value_counts(dropna=False).to_string())
print()
df_conflict = df[df['flag_type'] == 'rule_conflict'].copy()
print(f'rule_conflict flags: {len(df_conflict):,}  ({len(df_conflict)/n*100:.1f}%)')

if len(df_conflict):
    df_conf_eval = df_conflict.dropna(subset=['gt_key','llm_final_pred_key'])
    conf_acc = (df_conf_eval['llm_final_pred_key'] == df_conf_eval['gt_key']).mean() * 100
    print(f'L3 accuracy on conflict rows: {conf_acc:.1f}%  (vs overall {l3_acc:.1f}%)')
    cols = ['original_description','merchant_name','p1_bank_db_key','p2_ner_db_key',
            'llm_final_pred_key','gt_key','flag_detail','reason']
    print(df_conflict[[c for c in cols if c in df_conflict.columns]].head(15).to_string(index=False))

---
## 10. Confidence vs accuracy

In [ ]:
df_ca = df_eval.dropna(subset=['confidence','llm_final_pred_key','gt_key']).copy()
df_ca['l3_match'] = df_ca['llm_final_pred_key'] == df_ca['gt_key']
cs = df_ca.groupby('confidence').agg(count=('l3_match','size'), accuracy=('l3_match','mean')).reset_index()
cs['accuracy'] = (cs['accuracy']*100).round(1)
print(cs.to_string(index=False))

fig, ax = plt.subplots(figsize=(8,4))
ax2 = ax.twinx()
ax.bar(cs['confidence'], cs['count'], color='#3498db', alpha=0.6, label='Row count')
ax2.plot(cs['confidence'], cs['accuracy'], 'ro-', linewidth=2, markersize=6, label='L3 accuracy')
ax.set_xlabel('Confidence (1-10)'); ax.set_ylabel('Row count')
ax2.set_ylabel('L3 accuracy (%)'); ax2.set_ylim(0,100)
ax.set_title('Confidence calibration - Run 4', fontweight='bold')
lines1,labels1 = ax.get_legend_handles_labels()
lines2,labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'nb8_confidence_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Per-category accuracy

In [ ]:
df_cat = (
    df_l3.groupby('gt_key')
    .apply(lambda g: pd.Series({
        'total':    len(g),
        'correct':  (g['llm_final_pred_key'] == g['gt_key']).sum(),
        'accuracy': (g['llm_final_pred_key'] == g['gt_key']).mean() * 100,
    })).reset_index().sort_values('accuracy')
)
print('Bottom 15 categories:')
print(df_cat.head(15).to_string(index=False))
print()
print(f'Perfect (100%): {(df_cat.accuracy==100).sum()}')
print(f'Zero accuracy:  {(df_cat.accuracy==0).sum()}')

---
## 12. Per-provider accuracy

In [ ]:
if 'provider_name' in df.columns:
    df_prov = (
        df_l3.groupby('provider_name')
        .apply(lambda g: pd.Series({
            'total':    len(g),
            'correct':  (g['llm_final_pred_key'] == g['gt_key']).sum(),
            'accuracy': (g['llm_final_pred_key'] == g['gt_key']).mean() * 100,
        })).reset_index().sort_values('accuracy')
    )
    print(df_prov.to_string(index=False))
    fig, ax = plt.subplots(figsize=(8,5))
    colors = ['#e74c3c' if a < 50 else '#e67e22' if a < 65 else '#2ecc71' for a in df_prov['accuracy']]
    ax.barh(df_prov['provider_name'], df_prov['accuracy'], color=colors, height=0.7)
    ax.axvline(l3_acc,   color='black', linestyle='--', linewidth=1.2, label=f'Run 4 avg {l3_acc:.1f}%')
    ax.axvline(RUN3_L3,  color='grey',  linestyle=':',  linewidth=1.2, label=f'Run 3 avg {RUN3_L3}%')
    ax.set_xlabel('L3 Accuracy (%)')
    ax.set_title('Per-provider L3 accuracy - Run 4', fontweight='bold')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'nb8_provider_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('provider_name not in results - skipping')

---
## 13. Four-field accuracy comparison

Compares all prediction fields available in Run 4 on the rows where each fired.
p2_ner_llm_key is not available until Run 5.

| Field | Evaluated on |
|---|---|
| `naive_key` | All rows |
| `p1_bank_db_key` | P1 hit rows only |
| `p2_ner_db_key` | P2 hit rows only |
| `llm_final_pred_key` | All evaluable rows (and on P1/P2 hit subsets for delta) |

In [ ]:
def acc(subset, col):
    d = subset.dropna(subset=['gt_key', col])
    if not len(d): return None, 0
    return (d[col] == d['gt_key']).mean() * 100, len(d)

naive_acc,     naive_n    = acc(df,                'naive_key')
p1_db_acc,     p1_n       = acc(df[p1_hit],         'p1_bank_db_key')
p1_llm_acc,    _          = acc(df[p1_hit],         'llm_final_pred_key')
p2_db_acc,     p2_n       = acc(df[p2_hit],         'p2_ner_db_key')
p2_llm_acc,    _          = acc(df[p2_hit],         'llm_final_pred_key')
p2i_db_acc,    p2i_n      = acc(df[p1_miss_p2_hit], 'p2_ner_db_key')
p2i_llm_acc,   _          = acc(df[p1_miss_p2_hit], 'llm_final_pred_key')
final_acc,     final_n    = acc(df_eval,             'llm_final_pred_key')

def fmt(v):     return f'{v:.1f}%' if v is not None else 'n/a'
def delta(a,b): return f'({a-b:+.1f}pp vs DB)' if a is not None and b is not None else ''

print('=' * 70)
print('FOUR-FIELD ACCURACY COMPARISON')
print('=' * 70)
print(f'naive_key              (all rows,          n={naive_n:,}):  {fmt(naive_acc)}')
print()
print(f'p1_bank_db_key         (P1 hit rows,       n={p1_n:,}):  {fmt(p1_db_acc)}  <- DB direct')
print(f'llm_final_pred_key     (same P1 hit rows):              {fmt(p1_llm_acc)}  {delta(p1_llm_acc, p1_db_acc)}')
print()
print(f'p2_ner_db_key          (P2 hit rows,       n={p2_n:,}):  {fmt(p2_db_acc)}  <- DB direct')
print(f'llm_final_pred_key     (same P2 hit rows):              {fmt(p2_llm_acc)}  {delta(p2_llm_acc, p2_db_acc)}')
print()
print(f'p2_ner_db_key incr     (P1 miss + P2 hit,  n={p2i_n}):  {fmt(p2i_db_acc)}  <- DB direct')
print(f'llm_final_pred_key     (same incr rows):                {fmt(p2i_llm_acc)}  {delta(p2i_llm_acc, p2i_db_acc)}')
print()
print(f'llm_final_pred_key     (all eval rows,    n={final_n:,}):  {fmt(final_acc)}  <- primary metric')
print()
print('Delta interpretation:')
print('  Positive -> LLM adds value on top of DB hint')
print('  Negative -> LLM overriding correct DB hints (prompt tuning opportunity)')
print('  Near zero -> LLM faithfully following DB hint')

In [ ]:
labels = [
    'naive_key\n(all rows)',
    'p1_bank_db_key\n(P1 hit)',
    'llm_final\n(on P1 hit)',
    'p2_ner_db_key\n(P2 hit)',
    'llm_final\n(on P2 hit)',
    'llm_final_pred_key\n(all rows)',
]
values = [naive_acc, p1_db_acc, p1_llm_acc, p2_db_acc, p2_llm_acc, final_acc]
colors = ['#B4B2A9','#AFA9EC','#7F77DD','#9FE1CB','#1D9E75','#3C3489']

fig, ax = plt.subplots(figsize=(11,4))
bars = ax.bar(labels, values, color=colors, width=0.6)
for bar, v in zip(bars, values):
    if v: ax.text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
ax.set_title('Four-field accuracy comparison - Run 4', fontweight='bold')
ax.axhline(naive_acc, color='#888780', linestyle=':', linewidth=1, label=f'naive_key baseline {naive_acc:.1f}%')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'nb8_field_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# LLM override analysis - where LLM overrode a correct DB hint
for label, mask, db_col in [
    ('p1_bank_db_key', p1_hit, 'p1_bank_db_key'),
    ('p2_ner_db_key',  p2_hit, 'p2_ner_db_key'),
]:
    grp = df[mask].dropna(subset=['gt_key', db_col, 'llm_final_pred_key'])
    overrides = grp[
        (grp['llm_final_pred_key'] != grp[db_col]) &
        (grp[db_col] == grp['gt_key'])
    ]
    print(f'LLM overrode correct {label}: {len(overrides)} rows')
    if len(overrides):
        top = (overrides.groupby([db_col,'llm_final_pred_key'])
               .size().reset_index(name='count')
               .sort_values('count', ascending=False).head(10))
        print(top.to_string(index=False))
    print()

---
## 14. Ad-hoc query helpers

In [ ]:
# A) All mismatches for a specific GT category
errors = df_l3[df_l3['llm_final_pred_key'] != df_l3['gt_key']].copy()

GT_CAT = 'GROCERIES'
(
    errors[errors['gt_key'] == GT_CAT]
    [['original_description','merchant_name','p1_bank_db_key','p2_ner_db_key',
      'llm_final_pred_key','gt_key','execution_path','reason']]
    .head(20)
)

In [ ]:
# B) All rows where P1 or P2 hit - inspect DB signal quality
(
    df[p1_hit | p2_hit]
    [['original_description','merchant_name','p1_bank_db_key',
      'p2_llm_merchant','p2_ner_db_key',
      'llm_final_pred_key','gt_key','execution_path']]
    .head(30)
)

In [ ]:
# C) NER ran but no DB hit - what merchants did NER extract that aren't in the DB?
df_ner_miss = df[df['p2_llm_merchant'].notna() & (df['p2_ner_db_key'] == 'na')]
print(f'NER ran, no DB hit: {len(df_ner_miss)} rows')
print(df_ner_miss['p2_llm_merchant'].value_counts().head(30).to_string())

In [ ]:
# D) Free-text search across descriptions
SEARCH = 'woolworths'
(
    df[df['original_description'].str.lower().str.contains(SEARCH, na=False)]
    [['original_description','merchant_name','p1_bank_db_key','p2_ner_db_key',
      'llm_final_pred_key','gt_key','execution_path','reason']]
    .head(20)
)

In [ ]:
# E) L1 spend type errors
l1_errors = df_l1[df_l1['spend_type'] != df_l1['gt_spend_type']]
print(f'L1 errors: {len(l1_errors)}')
(
    l1_errors[['original_description','merchant_name','spend_type',
               'gt_spend_type','llm_final_pred_key','gt_key',
               'execution_path','reason']]
    .head(20)
)